In [0]:
from pyspark.sql.functions import current_timestamp
def add_ingestion_date(input_df):
    output_df = input_df.withColumn("ingestion_date", current_timestamp())
    return output_df

In [0]:
def overwrite_partition(db_name, table_name, column_partition, file_date):
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):
                spark.sql(f"DELETE FROM {db_name}.{table_name} WHERE {column_partition} = '{file_date}'")
    return 

In [0]:
from delta.tables import DeltaTable
def incremental_merge(db_name, table_name, df_name, condition, column_partition):
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):

        deltaTable = DeltaTable.forName(spark, f'{db_name}.{table_name}')

        deltaTable.alias('tgt') \
        .merge(
            df_name.alias('src'),
            f'{condition}'
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    else:

        df_name.write.mode("overwrite").partitionBy(f"{column_partition}").format("delta").saveAsTable(f"{db_name}.{table_name}")